# 📘 Notebook de Práctica
## Adquisición de Datos + Ciclo de Vida del Machine Learning

Este notebook integra dos temáticas de la materia:

1. **Unidad 3 — Adquisición de Datos**: tipos de datos, formatos tabulares (CSV, Excel, Parquet), consumo de APIs REST, autenticación, paginación, rate limits y control de calidad.
2. **El ciclo de vida del ML**: lectura robusta de archivos, EDA, limpieza de datos, y la regla de oro de la división train/test.

> 💡 Este notebook está **resuelto**: cada celda muestra el código funcionando con explicaciones detalladas. Úsalo como referencia y para seguir la clase paso a paso. El notebook de *Ejercicios* tiene la versión para que vos completes.

### 📂 Datasets que usa este notebook

Todos estos archivos deben estar en la carpeta `datasets/` (junto a este notebook):

| Archivo | Usado en | Descripción |
|---|---|---|
| `ventas_tienda.csv` | Sección 3 y 10 | Ventas de una tienda de electrónica (dataset "limpio") |
| `clientes_sucio.csv` | Sección 4 | Archivo con separador `;` y encoding `latin-1` |
| `config_api.json` | Sección 4 | Configuración de ejemplo en formato JSON |
| `calidad_datos.csv` | Secciones 5 y 6 | Dataset con nulos, duplicados e inconsistencias de texto |
| `estudiantes.csv` | Sección 7 | Dataset chico para ilustrar Data Leakage |

**Agenda**

| Sección | Tema |
|---|---|
| 1 | Setup del entorno |
| 2 | Tipos de datos y ciclo de vida analítico |
| 3 | Formatos tabulares: CSV, Excel, Parquet |
| 4 | Lectura robusta de archivos (encoding, separador, JSON) |
| 5 | EDA y control de calidad de datos |
| 6 | Limpieza de datos |
| 7 | División Train/Test y Data Leakage |
| 8 | Consumo de APIs REST con `requests` |
| 9 | Paginación y manejo de Rate Limits |
| 10 | Pipeline reproducible de punta a punta |


## 1. Setup del entorno

Importamos las librerías que vamos a usar en todo el notebook. Si falta alguna, instalala con `pip install <paquete>` (necesitarás `pandas`, `numpy`, `requests`, `openpyxl`, `pyarrow` y `scikit-learn`).

In [ ]:
# Librerías estándar
import os
import json
import time
from pathlib import Path

# Librerías de terceros
import numpy as np
import pandas as pd
import requests

# Scikit-learn (para la sección de train/test)
from sklearn.model_selection import train_test_split

# Configuración de pandas para que se vea prolijo
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# Carpeta donde están los datasets de este notebook
DATASETS = Path("datasets")
assert DATASETS.exists(), "No se encontró la carpeta 'datasets/'. Asegurate de descomprimirla junto al notebook."

print("Entorno listo ✅")
print("Archivos disponibles en datasets/:", [p.name for p in sorted(DATASETS.iterdir())])


**Sobre `.env` y credenciales (buenas prácticas)**

En un proyecto real, las claves de API **nunca** van escritas en el notebook ni se suben a un repositorio. Se guardan en un archivo `.env` (que se agrega a `.gitignore`, es decir, que Git ignora y nunca sube) y se leen con la librería `python-dotenv`:

```python
# archivo .env (no se sube al repositorio)
API_KEY=tu_api_key_aqui
```

```python
from dotenv import load_dotenv
load_dotenv()               # busca el archivo .env y carga sus variables
api_key = os.getenv("API_KEY")
```

¿Por qué tanto cuidado? Porque si subís una clave a un repositorio público (aunque sea por error), bots automáticos la detectan en minutos y pueden usarla para consumir tu cuota o generar cargos en tu cuenta. Más abajo (Sección 8) creamos un `.env` de ejemplo para practicar el patrón completo, sin usar una clave real.

## 2. Tipos de datos y ciclo de vida analítico

Recordá el ciclo que vimos en la Unidad de Ciclo de Vida del ML: **Definir problema → Adquisición → Limpieza → Análisis/Modelado → Comunicación/Producción**. Es un ciclo **iterativo**: los resultados de una etapa pueden obligarnos a volver a la anterior. Por ejemplo, si al limpiar los datos descubrimos que falta una columna clave, tenemos que volver a la etapa de Adquisición y conseguirla de otra fuente.

Este notebook se concentra especialmente en las dos primeras etapas (**Adquisición** y **Limpieza**), porque en la práctica son las que más tiempo insumen — se suele decir que un científico de datos dedica el 70-80% de su tiempo a esto, y solo el resto a modelar.

Antes de adquirir cualquier dato, conviene identificar de qué tipo es, porque eso determina qué herramientas usar:

| Tipo | Descripción | Ejemplos | Herramienta típica |
|---|---|---|---|
| Estructurados | Esquema fijo, organizados en tablas (filas y columnas) | Bases de datos SQL, CSV, Excel | `pandas`, SQL |
| Semi-estructurados | Tienen estructura, pero es flexible/anidada | JSON, XML, logs de servidores | `json`, `pandas.json_normalize` |
| No estructurados | Sin formato tabular predefinido | Texto libre, imágenes, audio, video | NLP, visión por computadora |
| Vectoriales | Representación numérica de datos complejos | Embeddings de texto o imágenes | Bases de datos vectoriales |

En este notebook vamos a trabajar principalmente con datos **estructurados** (CSV, Excel, Parquet) y **semi-estructurados** (JSON, respuestas de APIs).

Una idea que va a aparecer una y otra vez a lo largo del notebook, y que conviene tener siempre presente:

> **Garbage In, Garbage Out (GIGO):** un modelo — o cualquier análisis — es tan bueno como los datos con los que se lo alimenta. Ningún algoritmo, por sofisticado que sea, puede compensar datos mal adquiridos o mal limpiados.

## 3. Formatos tabulares: CSV, Excel y Parquet

Vamos a cargar un dataset real de ventas (`ventas_tienda.csv`), y a partir de él generar los otros dos formatos para poder compararlos. La idea es que veas con tus propios ojos las diferencias de tamaño y velocidad — no alcanza con memorizar la teoría.

In [ ]:
# 1) Cargamos el dataset original en formato CSV
ventas = pd.read_csv(DATASETS / "ventas_tienda.csv", parse_dates=["fecha"])

print(f"Filas: {len(ventas)} | Columnas: {list(ventas.columns)}")
ventas.head()


Este dataset ya viene "limpio" a propósito, porque el foco de esta sección es comparar **formatos de archivo**, no calidad de datos (eso lo vemos en las secciones 5 y 6). Ahora lo vamos a guardar en los otros dos formatos para poder compararlos.

- **CSV (Comma-Separated Values):** texto plano, sin tipos de datos ni compresión. Es el formato más universal: cualquier lenguaje o herramienta lo puede leer.
- **Excel (.xlsx):** formato binario propietario de Microsoft. Permite múltiples hojas, fórmulas, formato visual (colores, bordes) — ideal para reportes que va a abrir una persona, no un programa.
- **Parquet:** formato binario columnar (guarda los datos "por columna" en vez de "por fila"), con compresión y metadata de tipos incluida. Pensado para procesamiento a gran escala (lo usan Spark, motores de Big Data, data lakes).

In [ ]:
# Carpeta de salida (rutas relativas: buena práctica según pathlib)
salida_dir = Path("salidas")
salida_dir.mkdir(exist_ok=True)

ruta_csv = DATASETS / "ventas_tienda.csv"          # ya existe
ruta_xlsx = salida_dir / "ventas_tienda.xlsx"
ruta_parquet = salida_dir / "ventas_tienda.parquet"

# 2) Guardamos el mismo contenido en Excel y en Parquet
ventas.to_excel(ruta_xlsx, index=False)          # requiere openpyxl
ventas.to_parquet(ruta_parquet, index=False)     # requiere pyarrow

print("Archivos generados:")
for ruta in [ruta_csv, ruta_xlsx, ruta_parquet]:
    tamano_kb = ruta.stat().st_size / 1024
    print(f"  {ruta.name:<22} {tamano_kb:>8.1f} KB")


In [ ]:
# 3) Comparamos velocidad de lectura de cada formato (promedio de 5 lecturas)
import timeit

t_csv = timeit.timeit(lambda: pd.read_csv(ruta_csv), number=5) / 5
t_xlsx = timeit.timeit(lambda: pd.read_excel(ruta_xlsx), number=5) / 5
t_parquet = timeit.timeit(lambda: pd.read_parquet(ruta_parquet), number=5) / 5

comparativa = pd.DataFrame({
    "formato": ["CSV", "Excel", "Parquet"],
    "tamano_KB": [ruta_csv.stat().st_size / 1024, ruta_xlsx.stat().st_size / 1024, ruta_parquet.stat().st_size / 1024],
    "tiempo_lectura_seg": [t_csv, t_xlsx, t_parquet],
})
comparativa


**¿Por qué Parquet suele ganar en tamaño y velocidad?**

1. **Compresión columnar:** al guardar los datos columna por columna, valores repetidos (como `categoria` o `producto`, que tienen pocos valores distintos) se comprimen muchísimo mejor que en un CSV donde cada fila es independiente.
2. **Tipos de datos explícitos:** Parquet guarda el `dtype` de cada columna (si es entero, texto, fecha, etc.), así que al leerlo pandas no tiene que "adivinar" los tipos como hace con un CSV — eso ahorra tiempo de procesamiento.
3. **Lectura selectiva:** si sólo necesitás 2 de las 10 columnas de un archivo Parquet, podés leer *solo esas 2* sin cargar el resto a memoria. En CSV eso es imposible: hay que leer el archivo entero.

**Conclusión práctica — ¿cuándo usar cada uno?**

| Formato | Cuándo usarlo |
|---|---|
| CSV | Intercambio simple entre sistemas o personas, datasets chicos, máxima compatibilidad |
| Excel | Reportes para personas no técnicas, múltiples hojas, fórmulas, formato visual |
| Parquet | Pipelines de datos, almacenamiento a escala (Data Lakes), cuando la performance importa |


## 4. Lectura robusta de archivos (encoding, separador, JSON)

Los errores más comunes al leer archivos en un proyecto real son: **ignorar el encoding**, **asumir el separador** y **no cerrar el archivo correctamente**. Vamos a reproducirlos con un archivo real (`clientes_sucio.csv`) y corregirlos paso a paso.

**¿Qué es el encoding?** Es la manera en la que el texto se traduce a bytes (números) para guardarse en disco. `UTF-8` es el estándar moderno y soporta prácticamente cualquier caracter del mundo. `Latin-1` (o `ISO-8859-1`) es un encoding más viejo, típico de archivos generados por sistemas antiguos o exportados desde Excel en configuración regional de Argentina/España. Si leés un archivo `Latin-1` asumiendo que es `UTF-8` (o viceversa), los caracteres especiales (`ñ`, tildes) se rompen o directamente el archivo no se puede leer.

In [ ]:
# Antes de leer un archivo desconocido, SIEMPRE conviene mirar las primeras líneas "crudas"
with open(DATASETS / "clientes_sucio.csv", "rb") as f:
    primeras_lineas = f.readline() + f.readline()
print(primeras_lineas)


Al inspeccionar los bytes crudos ya podemos sospechar dos cosas: el separador no parece ser una coma, y hay bytes (`\xf3`, etc.) que no son ASCII simple — señal de que el encoding no es el que esperaríamos por defecto.

In [ ]:
# ❌ Lectura ingenua: asumimos coma y utf-8 -> falla o rompe los acentos
try:
    df_mal = pd.read_csv(DATASETS / "clientes_sucio.csv")  # sin sep ni encoding
    print(df_mal)
except UnicodeDecodeError as e:
    print("💥 Falló como esperábamos:")
    print(f"   {type(e).__name__}: {e}")
    print("   El archivo no está en UTF-8, y además el separador tampoco es una coma.")


In [ ]:
# ✅ Lectura robusta: especificamos separador y encoding correctos
df_bien = pd.read_csv(DATASETS / "clientes_sucio.csv", sep=";", encoding="latin-1")
df_bien


**Regla práctica:** antes de leer un archivo desconocido, siempre mirá las primeras líneas crudas (como hicimos arriba con `open(..., "rb")`) para detectar separador y encoding, en vez de asumir que siempre van a ser coma y UTF-8. Si el encoding sigue sin quedar claro, podés probar la librería `chardet`, que intenta detectarlo automáticamente:

```python
import chardet
with open("archivo.csv", "rb") as f:
    resultado = chardet.detect(f.read())
print(resultado)  # {'encoding': 'ISO-8859-1', 'confidence': 0.73, ...}
```

In [ ]:
# Lectura de JSON con el bloque `with` (context manager)
with open(DATASETS / "config_api.json", "r", encoding="utf-8") as f:
    config = json.load(f)

config


**¿Por qué usar `with` en vez de `open()` a secas?**

```python
# ❌ Riesgoso: si algo falla entre el open() y el close(), el archivo queda "colgado"
f = open("archivo.json")
datos = json.load(f)
f.close()

# ✅ Seguro: el bloque `with` garantiza que el archivo se cierre SIEMPRE,
#    incluso si ocurre un error en el medio
with open("archivo.json") as f:
    datos = json.load(f)
```

Un archivo abierto sin cerrar puede quedar bloqueado para otros procesos, o perder datos si el programa se corta abruptamente. El patrón `with` es la forma estándar y recomendada en Python para trabajar con cualquier recurso externo (archivos, conexiones de red, bases de datos).

## 5. EDA y control de calidad de datos

Antes de limpiar, **observamos**. Vamos a cargar `calidad_datos.csv`, un dataset con problemas típicos (nulos, duplicados, inconsistencias de texto), y aplicar el Análisis Exploratorio Inicial (EDA) que vimos en la clase de Ciclo de Vida del ML.

La secuencia recomendada siempre es la misma: **primero explorás, después decidís qué limpiar y cómo**. Limpiar "a ciegas" sin explorar antes es una receta segura para tomar malas decisiones (por ejemplo, eliminar una columna entera pensando que está vacía cuando en realidad solo tiene el 10% de nulos).

In [ ]:
datos_sucios = pd.read_csv(DATASETS / "calidad_datos.csv")
datos_sucios


In [ ]:
print("== df.info() : tipos de datos y cantidad de no-nulos ==")
datos_sucios.info()


In [ ]:
print("== df.isnull().sum() : cantidad exacta de nulos por columna ==")
print(datos_sucios.isnull().sum())
print()
print("== df.describe(include='all') : estadísticas rápidas ==")
datos_sucios.describe(include="all")


Con solo estas tres líneas ya identificamos varios problemas:
- `genero` tiene 1 valor nulo, y además mezcla mayúsculas/minúsculas (`"Macho"` y `"macho"` deberían ser el mismo valor).
- `edad` tiene 2 valores nulos.
- `ciudad` tiene 1 valor nulo.
- El `id=5` aparece dos veces — posible fila duplicada.

Vamos a formalizar esta inspección con una función reutilizable, que simula un **"Data Contract"**: un acuerdo formal (a veces literalmente un archivo de configuración en un proyecto real) sobre qué columnas debe tener un dataset y qué nivel de calidad es aceptable.

In [ ]:
def chequeo_de_esquema(df: pd.DataFrame, columnas_esperadas: list, umbral_nulos: float = 0.3) -> dict:
    '''Control de calidad mínimo: columnas presentes, tipos y % de nulos.

    Parámetros
    ----------
    df : DataFrame a auditar
    columnas_esperadas : lista de nombres de columna que el dataset DEBERÍA tener
    umbral_nulos : a partir de qué porcentaje de nulos consideramos que una columna es "problemática"

    Devuelve un diccionario-reporte con los hallazgos.
    '''
    reporte = {"columnas_faltantes": [], "columnas_con_muchos_nulos": [], "duplicados": 0}

    reporte["columnas_faltantes"] = [c for c in columnas_esperadas if c not in df.columns]

    for col in df.columns:
        pct_nulos = df[col].isnull().mean()
        if pct_nulos > umbral_nulos:
            reporte["columnas_con_muchos_nulos"].append((col, round(pct_nulos, 2)))

    reporte["duplicados"] = int(df.duplicated().sum())
    return reporte


chequeo_de_esquema(datos_sucios, columnas_esperadas=["id", "genero", "edad", "ciudad", "pais"])


El reporte nos dice que: falta la columna `pais` (rompe el *data contract* que definimos), y hay una fila duplicada. Ninguna columna supera el umbral de 30% de nulos, así que no hace falta descartar ninguna por completo. Vamos a corregir estos hallazgos en la próxima sección.

## 6. Limpieza de datos

Aplicamos las correcciones típicas: eliminar duplicados, normalizar texto (inconsistencias de mayúsculas/minúsculas) e imputar o eliminar nulos según el caso. Una idea importante: **cada decisión de limpieza es una decisión de negocio**, no solo una decisión técnica — y por eso siempre conviene documentarla (dejarla explicada en un comentario o en el README del proyecto).

In [ ]:
df_limpio = datos_sucios.copy()  # trabajamos sobre una copia, nunca sobre el original

# 1) Eliminar duplicados exactos
antes = len(df_limpio)
df_limpio = df_limpio.drop_duplicates()
print(f"Duplicados eliminados: {antes - len(df_limpio)}")


In [ ]:
# 2) Normalizar inconsistencias de texto (Macho/macho -> Masculino, Hembra/hembra -> Femenino)
#    .str.strip() elimina espacios de más, .str.lower() unifica mayúsculas/minúsculas antes de mapear
df_limpio["genero"] = df_limpio["genero"].str.strip().str.lower().map({
    "macho": "Masculino",
    "hembra": "Femenino",
})
df_limpio["genero"]


**¿Por qué no completamos el valor nulo de `genero`?** Porque sería *inventar* un dato sensible que no tenemos forma de inferir razonablemente. En cambio, para `edad` y `ciudad` sí podemos justificar una imputación:
- `edad`: usamos la **mediana** en vez del promedio, porque es más robusta ante valores extremos (outliers). Si hubiera una persona de 99 años en el dataset, el promedio se distorsionaría mucho más que la mediana.
- `ciudad`: al ser una categoría (no un número), no tiene sentido calcular una "mediana de ciudades". Usamos una categoría explícita `"Desconocida"` en vez de una ciudad real, para no introducir un dato falso.

In [ ]:
# 3) Imputar nulos: edad -> mediana, ciudad -> categoría "Desconocida"
df_limpio["edad"] = df_limpio["edad"].fillna(df_limpio["edad"].median())
df_limpio["ciudad"] = df_limpio["ciudad"].fillna("Desconocida")

df_limpio


In [ ]:
# Verificación final: ya no deberían quedar nulos en edad/ciudad, ni duplicados
assert df_limpio.duplicated().sum() == 0, "Todavía hay duplicados"
assert df_limpio["edad"].isnull().sum() == 0, "Todavía hay nulos en edad"
assert df_limpio["ciudad"].isnull().sum() == 0, "Todavía hay nulos en ciudad"
print("Dataset limpio ✅ (el nulo de 'genero' se dejó intacto a propósito, por ser un dato sensible que no podemos inferir)")


## 7. División Train/Test y Data Leakage

**Regla de oro:** nunca evaluamos el modelo con los mismos datos que usamos para entrenarlo. Y el orden importa: **primero dividimos, después limpiamos** (si limpiamos antes, se produce *Data Leakage*: información del set de prueba se filtra al de entrenamiento, y el modelo "hace trampa" sin que nos demos cuenta).

Vamos a trabajar con `estudiantes.csv`, un dataset chico a propósito para poder seguir el cálculo a mano y entender exactamente qué está pasando en cada paso.

In [ ]:
df_modelo = pd.read_csv(DATASETS / "estudiantes.csv")
df_modelo


In [ ]:
# ❌ INCORRECTO: imputamos con la media de TODO el dataset antes de dividir
media_global = df_modelo["horas_estudio"].mean()
df_leakage = df_modelo.copy()
df_leakage["horas_estudio"] = df_leakage["horas_estudio"].fillna(media_global)
print(f"❌ Media calculada con TODO el dataset (incluye datos que después van a ir a test): {media_global:.2f}")


¿Por qué está mal esto? Porque esa media incluye información de filas que, en un ratio 70/30, terminarían en el set de **test**. Es decir, estamos usando "el futuro" (los datos de evaluación) para tomar una decisión sobre "el presente" (el entrenamiento). El modelo entrenado así va a mostrar una performance en test artificialmente buena, que **no se va a replicar en producción** con datos nuevos — porque en producción no vamos a tener acceso a esa media "con trampa".

In [ ]:
# ✅ CORRECTO: primero dividimos, y la imputación se calcula SOLO con train
X = df_modelo[["horas_estudio"]]
y = df_modelo["aprobo"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

media_train = X_train["horas_estudio"].mean()
X_train = X_train.copy()
X_test = X_test.copy()
X_train["horas_estudio"] = X_train["horas_estudio"].fillna(media_train)
X_test["horas_estudio"] = X_test["horas_estudio"].fillna(media_train)  # el test se imputa con la estadística del TRAIN, nunca la propia

print(f"✅ Media calculada solo con TRAIN: {media_train:.2f}")
print(f"Tamaño train: {len(X_train)} | Tamaño test: {len(X_test)}")


> 🎓 **Analogía de clase:** evaluar con datos que el modelo ya vio es como darle a un estudiante un examen con las mismas preguntas ya resueltas en su libro: no aprendió, memorizó (**Overfitting**). Y usar estadísticas de todo el dataset para limpiar antes de dividir es como si ese mismo estudiante, antes del examen, hubiera visto "en promedio" cómo le fue al resto del curso en las preguntas — información que en un examen real nunca tendría disponible.

**¿Por qué `stratify=y`?** Le pedimos a `train_test_split` que mantenga la misma proporción de aprobados/no aprobados en train y en test que en el dataset original. Esto es especialmente importante cuando las clases están desbalanceadas (por ejemplo, 90% "No" y 10% "Sí"): sin `stratify`, podríamos terminar con un set de test que por azar no tenga ningún caso de la clase minoritaria, haciendo imposible evaluar bien el modelo en ese grupo.

## 8. Consumo de APIs REST con `requests`

Vamos a usar la **API pública de GitHub** (no requiere autenticación para consultas básicas) como ejemplo real de API REST. Una API REST expone "recursos" (por ejemplo, un repositorio) a través de URLs, y se interactúa con ellos usando verbos HTTP: `GET` (leer), `POST` (crear), `PUT`/`PATCH` (modificar), `DELETE` (borrar). En este notebook solo vamos a usar `GET`, porque solo necesitamos leer información pública.

Cada respuesta HTTP viene acompañada de un **código de estado** que resume qué pasó:

| Código | Significado |
|---|---|
| `200 OK` | Todo salió bien |
| `401 Unauthorized` | Falta autenticación o es inválida |
| `403 Forbidden` | No tenés permiso (o, en GitHub, se agotó el rate limit) |
| `404 Not Found` | El recurso no existe |
| `429 Too Many Requests` | Superaste el límite de solicitudes permitidas |
| `500 Internal Server Error` | Error del lado del servidor, no tuyo |


In [ ]:
url = "https://api.github.com/repos/pandas-dev/pandas"
respuesta = requests.get(url, timeout=10)

print("Status code:", respuesta.status_code)

if respuesta.status_code == 200:
    datos = respuesta.json()   # convierte el cuerpo de la respuesta (texto JSON) en un diccionario de Python
    print("Nombre:", datos["full_name"])
    print("⭐ Stars:", datos["stargazers_count"])
    print("Lenguaje principal:", datos["language"])
else:
    print("⚠️ La API no respondió 200. Puede ser un límite de tasa (rate limit) del entorno compartido.")
    print(respuesta.json())


**Autenticación (patrón, sin exponer claves):**

```python
headers = {"Authorization": "Bearer TU_TOKEN"}
requests.get(url, headers=headers)
```

Nunca escribas el token directamente en el notebook. Usamos `.env` + un pequeño loader (o `python-dotenv` en un proyecto real):

In [ ]:
# Creamos un archivo .env de ejemplo (en un proyecto real, este archivo NO se sube al repo:
# se agrega a .gitignore para que Git lo ignore)
ruta_env = Path(salida_dir) / ".env"
ruta_env.write_text("API_KEY=clave_de_ejemplo_12345\n")

# Cargamos las variables de entorno manualmente (sin dependencias externas, solo para ilustrar el patrón)
def cargar_env(ruta):
    variables = {}
    with open(ruta, "r") as f:
        for linea in f:
            linea = linea.strip()
            if linea and not linea.startswith("#") and "=" in linea:
                clave, valor = linea.split("=", 1)
                variables[clave] = valor
    return variables

env = cargar_env(ruta_env)
os.environ.update(env)
print("API_KEY cargada desde .env:", os.getenv("API_KEY"))

# 💡 En un proyecto real usarías: from dotenv import load_dotenv; load_dotenv()


## 9. Paginación y manejo de Rate Limits

Las APIs devuelven los resultados en "páginas" para no sobrecargar el servidor ni la red con respuestas gigantes, y limitan la cantidad de solicitudes por período (**Rate Limit**) para protegerse de abuso. Si superamos el límite, la API responde `429 Too Many Requests` (GitHub además puede devolver `403` cuando se trata de rate limit no autenticado).

In [ ]:
# Paginación por offset (page + per_page): le pedimos a la API "la página N, de tamaño per_page"
def obtener_paginas(url, params, max_paginas=3):
    resultados = []
    for pagina in range(1, max_paginas + 1):
        params_pagina = {**params, "page": pagina}
        r = requests.get(url, params=params_pagina, timeout=10)
        if r.status_code != 200:
            print(f"Se detuvo en la página {pagina}: status {r.status_code}")
            break
        items = r.json()
        if not items:  # página vacía = ya no hay más resultados
            break
        resultados.extend(items)
        print(f"Página {pagina}: {len(items)} ítems obtenidos")
    return resultados


repos = obtener_paginas(
    "https://api.github.com/orgs/pandas-dev/repos",
    params={"per_page": 5},
    max_paginas=2,
)
print(f"\nTotal de repos obtenidos: {len(repos)}")


> ⚠️ **Nota:** la API pública de GitHub permite solo 60 solicitudes/hora **sin autenticación**, compartidas entre todos los usuarios de la misma IP. Si al ejecutar este notebook ves `status 403` en vez de `200`, es probable que ese cupo ya esté agotado (por ejemplo, en un entorno compartido, como el aula). No es un error de tu código: es exactamente el escenario de *Rate Limit* que vamos a manejar a continuación con backoff exponencial. Con un token de autenticación (`headers={"Authorization": "Bearer TOKEN"}`) el límite sube a 5.000 solicitudes/hora.

**¿Qué es el "backoff exponencial"?** Es una estrategia de reintento: si una solicitud falla por un problema temporal (rate limit, error del servidor), en vez de reintentar inmediatamente (lo que probablemente vuelva a fallar y empeore la sobrecarga del servidor), esperamos un tiempo que se **duplica en cada intento** (1s, 2s, 4s, 8s...). Esto le da tiempo al servidor para recuperarse, y evita que miles de clientes reintenten todos al mismo tiempo (lo que se conoce como "thundering herd" o estampida).

In [ ]:
def get_con_backoff(url, params=None, headers=None, max_reintentos=4):
    '''Reintenta con backoff exponencial ante 429/403/500.'''
    delay = 1
    for intento in range(1, max_reintentos + 1):
        r = requests.get(url, params=params, headers=headers, timeout=10)

        if r.status_code == 200:
            return r

        if r.status_code in (429, 403, 500, 502, 503):
            restantes = r.headers.get("X-RateLimit-Remaining")
            print(f"Intento {intento}: status {r.status_code} "
                  f"(restantes según header: {restantes}). Esperando {delay}s...")
            time.sleep(delay)
            delay *= 2  # backoff exponencial: 1s -> 2s -> 4s -> 8s ...
        else:
            # Otro tipo de error (404, 401, etc.) -> no tiene sentido reintentar, no se va a "arreglar solo"
            return r

    return r  # devolvemos la última respuesta aunque haya fallado, para poder inspeccionarla


resp = get_con_backoff("https://api.github.com/repos/pandas-dev/pandas")
print("\nResultado final:", resp.status_code)


**Buenas prácticas para producción:**
- Monitorear los headers `X-RateLimit-Remaining` / `X-RateLimit-Reset` **antes** de que se agoten, para espaciar las solicitudes proactivamente en vez de esperar a que fallen.
- Usar paginación por **cursor** (un puntero opaco que la API te devuelve) en vez de offset (`page`) cuando los datos cambian en tiempo real — con offset, si se insertan o eliminan registros entre página y página, podés saltear o repetir resultados.
- Guardar resultados parciales en disco a medida que se obtienen, para no perder todo el progreso si la conexión se corta a mitad de camino.

## 10. Pipeline reproducible de punta a punta

Juntamos todo lo aprendido en una función de pipeline: **ingesta → validación de esquema → limpieza → guardado**. Esta es la estructura que verías en un `scripts/utils.py` o `pipeline.py` de un proyecto real de datos.

**¿Por qué "reproducible" importa tanto?** Porque un pipeline reproducible significa que, dado el mismo archivo de entrada, **siempre** vas a obtener el mismo resultado — sin pasos manuales, sin "lo corrí a mano en mi máquina y no sé bien qué hice". Eso permite que cualquier compañero de equipo (o vos mismo, dentro de 6 meses) pueda ejecutar el mismo proceso y confiar en el resultado.

In [ ]:
def pipeline_ventas(ruta_entrada: Path, ruta_salida: Path) -> pd.DataFrame:
    '''Pipeline mínimo reproducible sobre el dataset de ventas.

    Ingesta -> valida esquema -> limpia -> guarda versionado en Parquet.
    '''
    # 1. Ingesta
    df_crudo = pd.read_csv(ruta_entrada, parse_dates=["fecha"])

    # 2. Validación de esquema (control de calidad)
    columnas_esperadas = ["id_venta", "producto", "categoria", "cantidad", "precio_unitario", "fecha"]
    reporte = chequeo_de_esquema(df_crudo, columnas_esperadas)
    assert not reporte["columnas_faltantes"], f"Faltan columnas: {reporte['columnas_faltantes']}"

    # 3. Limpieza mínima
    df = df_crudo.drop_duplicates().copy()
    df = df.dropna(subset=["producto", "precio_unitario"])

    # 4. Chequeo/normalización de tipos
    df["cantidad"] = df["cantidad"].astype(int)
    df["total"] = df["cantidad"] * df["precio_unitario"]

    # 5. Guardado versionado (con timestamp, simulando un control de versiones simple de archivos)
    timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")
    ruta_final = ruta_salida / f"ventas_procesado_{timestamp}.parquet"
    df.to_parquet(ruta_final, index=False)

    print(f"Pipeline OK ✅ — {len(df)} filas guardadas en: {ruta_final.name}")
    return df


df_procesado = pipeline_ventas(DATASETS / "ventas_tienda.csv", salida_dir)
df_procesado.head()


## Resumen

- Los **datos** se clasifican en estructurados / semi-estructurados / no estructurados / vectoriales, y eso determina qué herramienta usar para leerlos.
- **CSV, Excel y Parquet** tienen usos distintos: portabilidad universal vs. riqueza de formato para humanos vs. rendimiento a escala para pipelines.
- Leer archivos "a ciegas" (sin revisar encoding/separador antes) es la causa más común de errores en la adquisición de datos — siempre conviene inspeccionar las primeras líneas crudas primero.
- El **EDA** (`info`, `describe`, `isnull().sum()`) siempre va antes de limpiar: no se puede tomar una buena decisión de limpieza sin entender primero qué hay realmente en los datos.
- **Nunca evaluamos con los datos de entrenamiento**, y el orden correcto es *dividir → limpiar*, no al revés (Data Leakage) — de lo contrario, la performance que medimos en test es un espejismo.
- Las **APIs REST** se consumen con `requests`; hay que manejar autenticación segura (`.env`), paginación (por offset o cursor) y rate limits (backoff exponencial).
- Un **pipeline reproducible** valida esquema, limpia, y guarda resultados versionados — así cualquier persona puede volver a ejecutarlo y obtener exactamente el mismo resultado.

➡️ Segui con el notebook **`Ejercicios_Adquisicion_y_Ciclo_de_Vida_ML.ipynb`** para practicar por tu cuenta, con datasets propios para cada ejercicio.